In [60]:
import numpy as np
import pandas as pd
import requests
from scipy.stats import norm
from datetime import datetime

BASE_URL = "https://thalex.com/api/v2/public"

In [75]:
def fetch_ticker(instrument_name: str) -> dict:
    url = f"{BASE_URL}/ticker"
    response = requests.get(url, params={"instrument_name": instrument_name})
    return response.json()["result"]

def fetch_instrument(instrument_name: str) -> dict:
    url = f"{BASE_URL}/instrument"
    response = requests.get(url, params={"instrument_name": instrument_name})
    return response.json()["result"]

def fetch_index(underlying: str = "BTCUSD") -> float:
    url = f"{BASE_URL}/index"
    response = requests.get(url, params={"underlying": underlying})
    return response.json()["result"]["price"]

In [76]:
# Sinclair's subjective pricing model (from the book)
# d3 = (ln(S/K) + (r + μ + σ²/2)T) / (σ√T)
# d4 = (ln(S/K) + (r + μ - σ²/2)T) / (σ√T)
# C = S·e^(μT)·N(d3) - K·e^(-rT)·N(d4)

def calc_d3(S, K, T, sigma, r, mu):
    return (np.log(S / K) + (r + mu + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def calc_d4(S, K, T, sigma, r, mu):
    return (np.log(S / K) + (r + mu - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def subjective_call_price(S, K, T, sigma, r, mu):
    d3 = calc_d3(S, K, T, sigma, r, mu)
    d4 = calc_d4(S, K, T, sigma, r, mu)
    return S * np.exp(mu * T) * norm.cdf(d3) - K * np.exp(-r * T) * norm.cdf(d4)

def subjective_put_price(S, K, T, sigma, r, mu):
    d3 = calc_d3(S, K, T, sigma, r, mu)
    d4 = calc_d4(S, K, T, sigma, r, mu)
    return K * np.exp(-r * T) * norm.cdf(-d4) - S * np.exp(mu * T) * norm.cdf(-d3)

In [77]:
# Fetch all available instruments
all_instruments = requests.get(f"{BASE_URL}/all_instruments").json()["result"]

# Filter to BTC options only
btc_options = [
    i for i in all_instruments 
    if i["product"] == "OBTCUSD"
]

# Show available expiries
expiries = sorted(set(i["instrument_name"].split("-")[1] for i in btc_options))
print("Available expiries:")
for exp in expiries:
    print(f"  {exp}")

Available expiries:
  01MAY26
  14APR26
  15APR26
  16APR26
  17APR26
  18APR26
  24APR26
  25DEC26
  25SEP26
  26JUN26
  29MAY26


In [79]:
# SELECT YOUR INSTRUMENT HERE
INSTRUMENT = "BTC-29MAY26-80000-C"

# Set your target adjusted forward (where you think price will be at expiry)
ADJUSTED_FORWARD = 80_000

print(f"Selected: {INSTRUMENT}")
print(f"Target adjusted forward: ${ADJUSTED_FORWARD:,.0f}")

Selected: BTC-29MAY26-80000-C
Target adjusted forward: $80,000


In [ ]:
# Fetch data
ticker = fetch_ticker(INSTRUMENT)
instrument = fetch_instrument(INSTRUMENT)
underlying = INSTRUMENT.split("-")[0]
spot = fetch_index(f"{underlying}USD")

# Extract parameters
strike = instrument["strike_price"]
expiration_ts = instrument["expiration_timestamp"]
option_type = instrument["option_type"]
iv = ticker["iv"]
mark_price = ticker["mark_price"]
forward = ticker["forward"]
delta = ticker["delta"]

# Time to expiration in years
now_ts = datetime.now().timestamp()
T = (expiration_ts - now_ts) / (365.25 * 24 * 60 * 60)

# Infer risk-free rate
r = np.log(forward / spot) / T

# Infer drift: if adjusted_forward = spot * e^((r+μ)T), then μ = ln(adjusted_forward/spot)/T - r
drift = np.log(ADJUSTED_FORWARD / spot) / T - r

# Calculate drift-adjusted values using original Sinclair formula
d3 = calc_d3(spot, strike, T, iv, r, drift)
d4 = calc_d4(spot, strike, T, iv, r, drift)
adjusted_delta = norm.cdf(d3)
adjusted_prob = norm.cdf(d4)

if option_type == "call":
    subjective_price = subjective_call_price(spot, strike, T, iv, r, drift)
else:
    subjective_price = subjective_put_price(spot, strike, T, iv, r, drift)

In [81]:
# Display results
print("=" * 48)
print(f"{INSTRUMENT}  —  Subjective Pricing")
print("=" * 48)
print(f"{'Spot price (S)':<32} {spot:>14,.0f}")
print(f"{'Forward price (F)':<32} {forward:>14,.0f}")
print(f"{'Strike price (K)':<32} {strike:>14,.0f}")
print(f"{'Implied volatility (σ)':<32} {iv*100:>13.1f}%")
print(f"{'Time to expiration (T)':<32} {T:>14.2f}")
print(f"{'Implied rate (r)':<32} {r*100:>13.1f}%")
print(f"{'Current delta':<32} {delta:>14.2f}")
print("-" * 48)
print(f"{'Current mark price':<32} ${mark_price:>13,.0f}")
print()
print(f"{'Adjusted forward':<32} {ADJUSTED_FORWARD:>14,.0f}")
print(f"{'Implied drift (μ)':<32} {drift*100:>13.1f}%")
print(f"{'Adjusted delta N(d3)':<32} {adjusted_delta:>14.2f}")
print(f"{'Adjusted probability N(d4)':<32} {adjusted_prob:>14.2f}")
print("-" * 48)
print(f"{'Drift adjusted price':<32} ${subjective_price:>13,.0f}")

BTC-29MAY26-80000-C  —  Subjective Pricing
Spot price (S)                           74,568
Forward price (F)                        74,632
Strike price (K)                         80,000
Implied volatility (σ)                    41.7%
Time to expiration (T)                     0.12
Implied rate (r)                           0.7%
Current delta                              0.34
------------------------------------------------
Current mark price               $        2,228

Adjusted forward                         80,000
Implied drift (μ)                         59.2%
Adjusted delta N(d3)                       0.53
Adjusted probability N(d4)                 0.47
------------------------------------------------
Drift adjusted price             $        4,551


In [84]:
# Edge calculation
edge = subjective_price - mark_price
edge_pct = (subjective_price / mark_price - 1) * 100

print()
print("=" * 48)
print(f"{'Edge (absolute)':<32} ${edge:>13,.0f}")
print(f"{'Edge (percentage)':<32} {edge_pct:>13.1f}%")
print("=" * 48)


Edge (absolute)                  $        2,324
Edge (percentage)                        104.3%


In [85]:
# Sensitivity analysis across adjusted forwards
adj_forwards = np.arange(forward * 0.8, forward * 1.6, forward * 0.1)
results = []

for adj_fwd in adj_forwards:
    mu = np.log(adj_fwd / spot) / T - r
    if option_type == "call":
        subj_price = subjective_call_price(spot, strike, T, iv, r, mu)
    else:
        subj_price = subjective_put_price(spot, strike, T, iv, r, mu)
    
    results.append({
        "Adj Forward": f"${adj_fwd:,.0f}",
        "Drift (μ)": f"{mu*100:.0f}%",
        "Subjective Price": f"${subj_price:,.0f}",
        "Edge vs Mark": f"${subj_price - mark_price:,.0f}",
        "Edge %": f"{(subj_price/mark_price - 1)*100:+.1f}%"
    })

df_sensitivity = pd.DataFrame(results)
print(f"\nSensitivity Analysis for {INSTRUMENT}")
print(f"Spot: ${spot:,.0f} | Forward: ${forward:,.0f} | Mark: ${mark_price:,.0f}")
print()
print(df_sensitivity.to_string(index=False))


Sensitivity Analysis for BTC-29MAY26-80000-C
Spot: $74,568 | Forward: $74,632 | Mark: $2,228

Adj Forward Drift (μ) Subjective Price Edge vs Mark   Edge %
    $59,706     -190%              $73      $-2,154   -96.7%
    $67,169      -90%             $558      $-1,670   -75.0%
    $74,632        0%           $2,226          $-2    -0.1%
    $82,096       81%           $5,733       $3,505  +157.4%
    $89,559      155%          $11,026       $8,798  +395.0%
    $97,022      224%          $17,520      $15,293  +686.5%
   $104,485      287%          $24,620      $22,392 +1005.2%
   $111,948      346%          $31,963      $29,736 +1334.9%


In [91]:
# Select expiry and show available strikes
EXPIRY = '29MAY26'

options_for_expiry = [
    i for i in btc_options 
    if EXPIRY in i["instrument_name"]
]

calls = sorted([i["instrument_name"] for i in options_for_expiry if i["option_type"] == "call"], 
               key=lambda x: int(x.split("-")[2]))
puts = sorted([i["instrument_name"] for i in options_for_expiry if i["option_type"] == "put"],
              key=lambda x: int(x.split("-")[2]))

print(f"Calls for {EXPIRY}:")
for c in calls:
    print(f"  {c}")
print(f"\nPuts for {EXPIRY}:")
for p in puts:
    print(f"  {p}")

Calls for 29MAY26:
  BTC-29MAY26-50000-C
  BTC-29MAY26-60000-C
  BTC-29MAY26-65000-C
  BTC-29MAY26-68000-C
  BTC-29MAY26-70000-C
  BTC-29MAY26-72000-C
  BTC-29MAY26-75000-C
  BTC-29MAY26-77000-C
  BTC-29MAY26-80000-C
  BTC-29MAY26-85000-C
  BTC-29MAY26-90000-C
  BTC-29MAY26-100000-C

Puts for 29MAY26:
  BTC-29MAY26-50000-P
  BTC-29MAY26-60000-P
  BTC-29MAY26-65000-P
  BTC-29MAY26-68000-P
  BTC-29MAY26-70000-P
  BTC-29MAY26-72000-P
  BTC-29MAY26-75000-P
  BTC-29MAY26-77000-P
  BTC-29MAY26-80000-P
  BTC-29MAY26-85000-P
  BTC-29MAY26-90000-P
  BTC-29MAY26-100000-P


In [92]:
from time import sleep

# Fetch all tickers for the expiry upfront
all_options = calls + puts
tickers = {}
instruments = {}

for inst_name in all_options:
    tickers[inst_name] = fetch_ticker(inst_name)
    instruments[inst_name] = fetch_instrument(inst_name)
    sleep(0.05)

# Calculate for all strikes
results_all = []

for inst_name in all_options:
    tick = tickers[inst_name]
    inst = instruments[inst_name]
    
    k = inst["strike_price"]
    opt_type = inst["option_type"]
    sigma = tick["iv"]
    mp = tick["mark_price"]
    fwd = tick["forward"]
    
    r_inst = np.log(fwd / spot) / T
    mu_inst = np.log(ADJUSTED_FORWARD / spot) / T - r_inst
    
    if opt_type == "call":
        subj = subjective_call_price(spot, k, T, sigma, r_inst, mu_inst)
    else:
        subj = subjective_put_price(spot, k, T, sigma, r_inst, mu_inst)
    
    edge = subj - mp
    edge_pct = (subj / mp - 1) * 100 if mp > 0 else 0
    
    results_all.append({
        "instrument": inst_name,
        "strike": k,
        "type": opt_type,
        "mark_price": mp,
        "subjective_price": subj,
        "edge": edge,
        "edge_pct": edge_pct,
        "delta": tick["delta"]
    })

df_all = pd.DataFrame(results_all)
df_all

,instrument,strike,type,mark_price,subjective_price,edge,edge_pct,delta
0,BTC-29MAY26-50000-C,50000.0,call,25173.877002,29886.558840,4712.681838,18.720525,0.976950
1,BTC-29MAY26-60000-C,60000.0,call,15634.031250,20161.578150,4527.546900,28.959562,0.909346
2,BTC-29MAY26-65000-C,65000.0,call,11204.242584,15468.909525,4264.666940,38.062965,0.832032
3,BTC-29MAY26-68000-C,68000.0,call,8804.460088,12802.437267,3997.977179,45.408545,0.760830
4,BTC-29MAY26-70000-C,70000.0,call,7366.063933,11130.438232,3764.374299,51.104285,0.702195
5,BTC-29MAY26-72000-C,72000.0,call,6049.637035,9543.609937,3493.972902,57.755083,0.636938
6,BTC-29MAY26-75000-C,75000.0,call,4373.683808,7396.496756,3022.812948,69.113660,0.529964
7,BTC-29MAY26-77000-C,77000.0,call,3443.455746,6126.081251,2682.625504,77.905038,0.456746
8,BTC-29MAY26-80000-C,80000.0,call,2345.488461,4510.112617,2164.624156,92.288843,0.351810
9,BTC-29MAY26-85000-C,85000.0,call,1155.538591,2539.523514,1383.984923,119.769684,0.207196


In [ ]:
# Calculate N(d2) for each option (risk-neutral probability of ITM)
def calc_d2(S, K, T, sigma, r):
    """Standard d2: (ln(S/K) + (r - sigma^2/2) * T) / (sigma * sqrt(T))"""
    return (np.log(S / K) + (r - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

df_all["d2"] = df_all.apply(
    lambda row: calc_d2(spot, row["strike"], T, tickers[row["instrument"]]["iv"], 
                        np.log(tickers[row["instrument"]]["forward"] / spot) / T), 
    axis=1
)
df_all["Nd2"] = df_all.apply(
    lambda row: norm.cdf(row["d2"]) if row["type"] == "call" else norm.cdf(-row["d2"]),
    axis=1
)

df_all[["instrument", "strike", "type", "delta", "Nd2", "mark_price", "subjective_price", "edge_pct"]].sort_values("strike")

In [ ]:
import altair as alt

df_calls = df_all[df_all["type"] == "call"].sort_values("strike")
df_puts = df_all[df_all["type"] == "put"].sort_values("strike")

zero_line = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#666666", strokeDash=[4, 2]).encode(y="y:Q")

# Calls
calls_edge = (
    alt.Chart(df_calls, width=400, height=300)
    .mark_bar(color="steelblue", opacity=0.9)
    .encode(
        x=alt.X("strike:O", title="Strike", sort="ascending"),
        y=alt.Y("edge:Q", title="Edge ($)"),
        tooltip=["instrument", "strike", "edge", "edge_pct", "mark_price"]
    )
)

calls_mark = (
    alt.Chart(df_calls)
    .mark_circle(color="white", size=50, opacity=0.9)
    .encode(
        x=alt.X("strike:O", sort="ascending"),
        y=alt.Y("mark_price:Q"),
        tooltip=["instrument", "strike", "mark_price"]
    )
)

# Puts
puts_edge = (
    alt.Chart(df_puts, width=400, height=300)
    .mark_bar(color="coral", opacity=0.9)
    .encode(
        x=alt.X("strike:O", title="Strike", sort="ascending"),
        y=alt.Y("edge:Q", title="Edge ($)"),
        tooltip=["instrument", "strike", "edge", "edge_pct", "mark_price"]
    )
)

puts_mark = (
    alt.Chart(df_puts)
    .mark_circle(color="white", size=50, opacity=0.9)
    .encode(
        x=alt.X("strike:O", sort="ascending"),
        y=alt.Y("mark_price:Q"),
        tooltip=["instrument", "strike", "mark_price"]
    )
)

(
    (calls_edge + calls_mark + zero_line).resolve_scale(y="independent").properties(title=f"Calls - Adj Forward: ${ADJUSTED_FORWARD:,.0f}") 
    | (puts_edge + puts_mark + zero_line).resolve_scale(y="independent").properties(title=f"Puts - Adj Forward: ${ADJUSTED_FORWARD:,.0f}")
).configure(
    background="black",
    padding={"right": 30, "left": 30, "top": 40, "bottom": 30},
).configure_axis(
    domain=False,
    grid=False,
    labelColor="white",
    labelFontSize=10,
    labelPadding=10,
    ticks=False,
    titleColor="white",
    titleFontSize=12,
    titlePadding=10,
).configure_title(
    color="white",
    fontSize=14,
).configure_view(
    stroke=None
)

In [95]:
# Highest relative edge (edge %)
df_all.sort_values("edge_pct", ascending=False)[
    ["instrument", "strike", "type", "delta", "mark_price", "subjective_price", "edge", "edge_pct"]
].head(10).style.format({
    "mark_price": "${:,.0f}",
    "subjective_price": "${:,.0f}",
    "edge": "${:,.0f}",
    "edge_pct": "{:+.1f}%",
    "delta": "{:.2f}",
    "strike": "{:,.0f}"
})

,instrument,strike,type,delta,mark_price,subjective_price,edge,edge_pct
11,BTC-29MAY26-100000-C,"100,000",call,0.03,$100,$319,$219,+219.4%
10,BTC-29MAY26-90000-C,"90,000",call,0.11,$543,"$1,353",$810,+149.2%
9,BTC-29MAY26-85000-C,"85,000",call,0.21,"$1,156","$2,540","$1,384",+119.8%
8,BTC-29MAY26-80000-C,"80,000",call,0.35,"$2,345","$4,510","$2,165",+92.3%
7,BTC-29MAY26-77000-C,"77,000",call,0.46,"$3,443","$6,126","$2,683",+77.9%
6,BTC-29MAY26-75000-C,"75,000",call,0.53,"$4,374","$7,396","$3,023",+69.1%
5,BTC-29MAY26-72000-C,"72,000",call,0.64,"$6,050","$9,544","$3,494",+57.8%
4,BTC-29MAY26-70000-C,"70,000",call,0.70,"$7,366","$11,130","$3,764",+51.1%
3,BTC-29MAY26-68000-C,"68,000",call,0.76,"$8,804","$12,802","$3,998",+45.4%
2,BTC-29MAY26-65000-C,"65,000",call,0.83,"$11,204","$15,469","$4,265",+38.1%
